# ODI to Databricks Migration

## Source File: `TARGET`
**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook contains the converted Spark SQL for the ODI scenario, performing an insert into the `TRG_LOC` table from `LOCATIONS`.

In [ ]:
import datetime

dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (e.g., F for Full, I for Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

# Defaulting current_timestamp to a static value for consistent runs if not dynamically provided.
# In a real scenario, this would dynamically reflect the current job start time.
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "Current ETL Extract Timestamp")

# ETL Parameters
The following parameters control the ETL execution:

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
    '${ETL_PROC_WID}' AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO,
    '${ETL_CURRENT_EXTRACT_TIME}' AS ETL_CURRENT_EXTRACT_TIME
"""))

# Target Table: `workspace.hr.trg_loc`

## SCEN_TASK_NO: Implicit DDL for Target Table

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
) USING DELTA;

## SCEN_TASK_NO: Insert into Target (Conversion of Original INSERT)

In [ ]:
%sql
INSERT INTO workspace.hr.trg_loc
(
  LOCATION_ID ,
  STREET_ADDRESS ,
  POSTAL_CODE ,
  CITY ,
  STATE_PROVINCE ,
  COUNTRY_ID
)
SELECT
  LOCATIONS.LOCATION_ID ,
  LOCATIONS.STREET_ADDRESS ,
  LOCATIONS.POSTAL_CODE ,
  LOCATIONS.CITY ,
  LOCATIONS.STATE_PROVINCE ,
  LOCATIONS.COUNTRY_ID
FROM
  workspace.hr.locations AS LOCATIONS;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_loc FROM workspace.hr.trg_loc;

# Conversion Notes

*   **Empty SCEN_TASK_NOs**: The original file contained empty `SCEN_TASK_NO` blocks ({10}, {20}, {30}). These have been omitted from the notebook as they did not contain executable SQL statements.
*   **Oracle Hints Removed**: The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Databricks Delta Lake.
*   **Schema and Table Naming**: Oracle schema `HR` and table names `TRG_LOC`, `LOCATIONS` have been converted to lowercase and prefixed with `workspace.` as per naming conventions (e.g., `HR.TRG_LOC` -> `workspace.hr.trg_loc`).
*   **Implicit DDL**: A `CREATE TABLE IF NOT EXISTS` statement has been added for `workspace.hr.trg_loc` based on the columns present in the `INSERT` statement. Data types (`BIGINT`, `STRING`) have been inferred based on common usage for HR schema tables. If precise DDL for `TRG_LOC` or `LOCATIONS` is available, these types should be verified and adjusted.